# Import Libraries

In [0]:
# Use Case: Classify news headlines into Politics, Business, Sports, Entertainment

# ---------------------------------------------------------------------
# 1️⃣ Import Required Libraries
# ---------------------------------------------------------------------
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Load the Dataset

In [0]:
%python
# Load the Databricks table as a Spark DataFrame
df_spark = spark.read.table("bigdata.default.news_headlines_dataset")

# Convert to pandas DataFrame if needed
df = df_spark.toPandas()

print("✅ Dataset Loaded Successfully!")
print("Shape:", df.shape)
display(df_spark.head())

✅ Dataset Loaded Successfully!
Shape: (104, 2)


Row(Headline='President signs new climate bill into law', Category='Politics')

In [0]:
print(df)

                                              Headline       Category
0            President signs new climate bill into law       Politics
1     Tech giant stock hits all-time high after merger       Business
2         Team wins championship with last-second goal         Sports
3     New blockbuster movie smashes box office records  Entertainment
4           Senate debates key policy reform this week       Politics
..                                                 ...            ...
99      Director releases surprise album, fans go wild  Entertainment
100        Senator debates key policy reform this week       Politics
101  Financial group stock hits all-time high after...       Business
102      Squad wins championship with last-second goal         Sports
103     Pop star releases surprise album, fans go wild  Entertainment

[104 rows x 2 columns]


# Data Preprocessing

In [0]:
# Data Preprocessing
# ---------------------------------------------------------------------
# Check for missing values
print("\nChecking for missing values:")
print(df.isnull().sum())

# Convert headlines to string (safety check)
df["Headline"] = df["Headline"].astype(str)


Checking for missing values:
Headline    0
Category    0
dtype: int64


# Feature Extraction

In [0]:
# 4️⃣ Feature Extraction using TF-IDF
# ---------------------------------------------------------------------
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=5000)
X = vectorizer.fit_transform(df["Headline"])
y = df["Category"]


# Splitting of the Data

In [0]:
# 5️⃣ Split Data into Train and Test Sets (70:30 ratio)
# ---------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training Samples: {X_train.shape[0]}, Testing Samples: {X_test.shape[0]}")


Training Samples: 72, Testing Samples: 32


# Model Building and Evaluation

In [0]:
# 6️⃣ Build and Evaluate Multiple Models
# ---------------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Linear SVM": LinearSVC(max_iter=5000)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"\n📊 Model: {name}")
    print("Accuracy:", round(acc, 4))
    print("Classification Report:\n", classification_report(y_test, preds, zero_division=0))
    results[name] = acc


📊 Model: Logistic Regression
Accuracy: 0.75
Classification Report:
                precision    recall  f1-score   support

     Business       1.00      0.50      0.67         8
Entertainment       0.80      1.00      0.89         8
     Politics       0.60      0.75      0.67         8
       Sports       0.75      0.75      0.75         8

     accuracy                           0.75        32
    macro avg       0.79      0.75      0.74        32
 weighted avg       0.79      0.75      0.74        32


📊 Model: Naive Bayes
Accuracy: 0.75
Classification Report:
                precision    recall  f1-score   support

     Business       1.00      0.50      0.67         8
Entertainment       1.00      1.00      1.00         8
     Politics       0.50      0.75      0.60         8
       Sports       0.75      0.75      0.75         8

     accuracy                           0.75        32
    macro avg       0.81      0.75      0.75        32
 weighted avg       0.81      0.75      

In [0]:
# 7️⃣ Compare Model Performance
# ---------------------------------------------------------------------
best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]
print("\n✅ Best Performing Model:", best_model_name, "with accuracy =", round(best_accuracy, 4))

# Define best_model based on the best_model_name
best_model = models[best_model_name]



✅ Best Performing Model: Logistic Regression with accuracy = 0.75


# Testing of Model

In [0]:
# 8️⃣ Predict on New Headlines Dynamically
# ---------------------------------------------------------------------
# You can replace these headlines with any new ones for real-time testing
new_headlines = [
    "Government passes new trade agreement",
    "Actor wins award for best performance",
    "Stock market shows record growth",
    "Team secures victory in world cup final"
]

new_vect = vectorizer.transform(new_headlines)
new_preds = best_model.predict(new_vect)

for h, p in zip(new_headlines, new_preds):
    print(f"📰 '{h}' --> Predicted Category: {p}")

📰 'Government passes new trade agreement' --> Predicted Category: Politics
📰 'Actor wins award for best performance' --> Predicted Category: Sports
📰 'Stock market shows record growth' --> Predicted Category: Business
📰 'Team secures victory in world cup final' --> Predicted Category: Sports
